# Platonic Earth: the full website, running in Colab

This notebook runs our **backend and website** in Colab. You can then pick **any place on Earth**, set a **radius**, and search the live Sentinel-2 imagery inside it in plain English, for example *"parking lots and big roofs"*.

**Nothing is trained here.** The backend loads the rotation we fitted on an RTX 5060 Ti, the **OSM-Sentinel 1M rotation**: all 1,000,000 image–caption pairs from 331 regions in 117 countries, file `colab/artifacts/osm_1m_rotation.npz`, SHA-256 `358c5b94aa58…`. Our search benchmark measured the earlier fit, which left the 50 test regions out. It uses the same two frozen public models, DINOv2-small and all-MiniLM-L6-v2.

**Run it:** *Runtime → Run all.* It takes about 3–5 minutes the first time. A GPU runtime is faster but not required. Each step checks itself and stops with a clear message if something is wrong.

| Step | What it does |
|---|---|
| 1 | Get the code (branch `colab`) |
| 2 | Check that Colab can reach the data services |
| 3 | Install the dependencies |
| 4 | Put the fitted artifacts in place (checksum-verified, no training) |
| 5 | Download the two frozen models |
| 6 | Start the backend and confirm it runs the 1M rotation |
| 7 | End-to-end test: analyse a small live area and search it |
| 8 | Open the website |

*Prototype, not a land-use determination.*

## 1 · Get the code

In [ ]:
import os, sys, json, time, subprocess, urllib.request, urllib.error, urllib.parse

REPO, BRANCH = "https://github.com/JohnPoole3rd/veritas.git", "colab"
WORK = os.environ.get("PE_WORK") or ("/content/veritas" if os.path.isdir("/content") else os.path.abspath("veritas"))
PORT = int(os.environ.get("PE_PORT", "8000"))
ROTATION = "colab/artifacts/osm_1m_rotation.npz"
ROTATION_SHA12 = "358c5b94aa58"

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(f"Step failed (exit {r.returncode}): {cmd}")

if not os.path.isdir(os.path.join(WORK, ".git")):
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO} {WORK}")
os.chdir(WORK)
sh("git log --oneline -1")

## 2 · Check that Colab can reach the data services
The website downloads satellite imagery live. Any HTTP answer, even an error code, counts as reachable; only a network failure stops the notebook.

In [ ]:
SERVICES = {
    "GitHub": "https://github.com",
    "Hugging Face (models)": "https://huggingface.co",
    "Earth Search (Sentinel-2 catalogue)": "https://earth-search.aws.element84.com/v1",
    "Sentinel-2 imagery on AWS": "https://sentinel-cogs.s3.us-west-2.amazonaws.com/",
    "Planetary Computer (Landsat heat)": "https://planetarycomputer.microsoft.com/api/stac/v1",
    "Basemap (Esri)": "https://server.arcgisonline.com/ArcGIS/rest/services?f=json",
}
bad = []
for name, url in SERVICES.items():
    try:
        code_ = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "platonic-earth-colab"}), timeout=20).status
    except urllib.error.HTTPError as e:
        code_ = e.code  # the service answered
    except Exception as e:
        code_ = None; bad.append(f"{name}: {type(e).__name__}: {e}")
    print(f"{'ok ' if code_ else 'FAIL'}  {name}  ({code_})")
if bad:
    raise SystemExit("Colab cannot reach: " + "; ".join(bad))

## 3 · Install the dependencies (about 1–2 minutes)

In [ ]:
if os.environ.get("PE_SKIP_PIP") != "1":
    sh(f"{sys.executable} -m pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU works, just slower)")

## 4 · Put the fitted artifacts in place
This verifies the 1M rotation's checksum and copies three small precomputed embedding files to where the backend reads them. Nothing is trained.

In [ ]:
sh(f"{sys.executable} colab/prepare.py")

## 5 · Download the two frozen models once

In [ ]:
sh(f"{sys.executable} app.py --prefetch")

## 6 · Start the backend with the 1M OSM rotation

In [ ]:
def api(path, body=None, timeout=60):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}", data=data,
                                 headers={"Content-Type": "application/json"} if data else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

env = dict(os.environ, PE_ROTATION=ROTATION)
server_log = open(os.path.join(WORK, "server.log"), "w")
server = subprocess.Popen([sys.executable, "-m", "web.server", "--port", str(PORT), "--host", "127.0.0.1", "--no-classic", "--online"],
                          cwd=WORK, env=env, stdout=server_log, stderr=subprocess.STDOUT)
meta = None
for _ in range(150):
    if server.poll() is not None:
        break
    try:
        meta = api("/api/meta", timeout=5); break
    except Exception:
        time.sleep(2)
if meta is None:
    print(open(os.path.join(WORK, "server.log")).read()[-4000:])
    raise SystemExit("The backend did not start; its log is printed above.")
print("mode:", meta["mode"], "| device:", meta["device"])
print("rotation:", meta["rotation"])
if meta["mode"] != "REAL" or not meta.get("rotation") or meta["rotation"]["sha256"] != ROTATION_SHA12:
    raise SystemExit("The backend is not running the 1M OSM rotation; stopping.")
print("Backend is up at port", PORT)

## 7 · End-to-end test: analyse a small live area and search it
A ~1 km box in Upper Arlington, Ohio, our test town. This downloads the clearest recent Sentinel-2 pass, labels every 640 m square with the 1M rotation, and runs one plain-English search. It takes about 15–60 s.

In [ ]:
lat, lon, half_km = 40.0275, -83.0735, 1.0
dlat, dlon = half_km / 110.574, half_km / (111.320 * __import__("math").cos(__import__("math").radians(lat)))
bbox = [round(lon - dlon, 5), round(lat - dlat, 5), round(lon + dlon, 5), round(lat + dlat, 5)]
print("estimate:", api("/api/region/estimate", {"bbox": bbox})["message"])
t0 = time.time()
job = api("/api/region", {"bbox": bbox, "when": None})["job"]
while True:
    j = api(f"/api/jobs/{job}")
    if j["status"] != "running":
        break
    time.sleep(2)
if j["status"] != "done":
    raise SystemExit(f"Region analysis failed: {j.get('error')}")
region = j["result"]
print(f"analysed {region['n_tiles']} squares from {region['scene']['id']} in {time.time() - t0:.1f} s")
query = "houses and residential streets"
res = api(f"/api/region/{region['id']}/search?q={urllib.parse.quote(query)}&k=5")
print(f"top matches for '{query}':")
for t in res["top"]:
    print(f"  square #{t['i']:>3}  model label: {region['tiles'][t['i']]['cls']:<22} similarity {t['score']:.3f}")
print("End-to-end test passed.")

## 8 · Open the website
Click the link below to open it in a new tab. Scroll to **Search anywhere**, search a place, press **Radius**, then **Analyse area**, then use the **Ask** tab.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, path="/")
except ImportError:
    print(f"Not running in Colab: open http://localhost:{PORT}/ in your browser.")

Run the next cell to show the website inside this notebook instead.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, path="/", height=900)
except ImportError:
    print(f"Not running in Colab: open http://localhost:{PORT}/ in your browser.")

### If something goes wrong
- **Backend log:** run `print(open('server.log').read()[-4000:])` in a new cell.
- **Session limits:** the website runs only while this Colab session is alive. If Colab disconnects, run all cells again; areas you analysed are re-downloaded.
- **Area size:** a study area is limited to 400 squares of 640 m, so a radius of 6 km or less fits.
- **Colab's free tier** may stop long-running web sessions. For judges, the public static site and the demo video are the reliable way to see the project.
- **Which model the numbers come from:** the site's "Results" chapter shows our logged EuroSAT-model numbers. The search uses the 1M OSM rotation, as confirmed in step 6.